# Assignment - Question 1: AI-Powered Resume Screening Assistant
**Applied Generative AI & Agentic Systems Coursework**  
**Student Submission:** AI Resume Screening System  
**Frameworks:** LangChain, FAISS, Pydantic, OpenRouter (`openrouter/free`)  
**Assignment Track:** Document Intelligence & RAG  

---

## 1. Project Background & RAG System Architecture

In modern recruitment workflows, talent acquisition teams receive hundreds of resumes for technical openings. Manual screening is time-consuming and often inconsistent. This project implements an **AI-powered Resume Screening Assistant** using **LangChain** and **Retrieval-Augmented Generation (RAG)** to evaluate candidates against a target Job Description (JD).

### System Pipeline:
```
[Uploaded Resume (PDF)] 
         │
         ▼
[PyPDFLoader] ────────► Extracts raw text pages
         │
         ▼
[RecursiveCharacterTextSplitter] ───► Chunks of 400 chars (overlap: 50)
         │
         ▼
[Normalized Feature Embeddings] ───► 512-dim normalized vector representation
         │
         ▼
[FAISS Vector Store] ◄─── Indexed for cosine/L2 nearest neighbor search
         │
    [Job Description Query]
         │
         ▼
[Retriever (top-k=4)] ───► Most relevant resume sections
         │
         ▼
[Prompt Template + OpenRouter LLM] ───► openrouter/free with grounded context
         │
         ▼
[JsonOutputParser (Pydantic)] ───► Structured Candidate Evaluation
```

### Key Design Principles:
1. **Strict Context Grounding:** The assistant is prompted to base its evaluations *solely* on the retrieved text from the candidate's resume, mitigating hallucinations.
2. **Deterministic Structured Output:** Rather than conversational prose, the system outputs validated JSON matching our `CandidateEvaluation` schema (scores, skills, strengths, weaknesses, recommendation).
3. **Local Embedding Vector Store:** By utilizing a normalized local feature embedding integrated into LangChain, we maintain high throughput, local reproducibility, and zero external embedding costs.

## 2. Environment Setup & Dependency Imports

Let us import the necessary LangChain modules, document loaders, text splitters, vector stores, and Pydantic schemas.

In [1]:
import os
import json
import numpy as np
import pandas as pd
from typing import List
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import HashingVectorizer

# LangChain modules
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

print("All necessary libraries imported successfully!")

C:\Users\USSER\AppData\Local\Temp\ipykernel_11472\2504588747.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
g:\ML Deployment\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All necessary libraries imported successfully!


## 3. LLM Configuration (OpenRouter) & Local Embeddings

For this assignment, I am using **OpenRouter** as the LLM provider using the `openrouter/free` model via its OpenAI-compatible endpoint. 

To keep the pipeline efficient, reproducible, and free of embedding rate limits, I implemented `StudentFeatureEmbeddings` using LangChain's `Embeddings` interface. It transforms text chunks into normalized 512-dimensional vectors using feature hashing, which FAISS indexes for rapid similarity retrieval.

In [ ]:
class StudentFeatureEmbeddings(Embeddings):
    """
    A fast, local feature embedding class implementing LangChain's Embeddings interface.
    Generates normalized 512-dimensional dense embeddings for FAISS.
    """
    def __init__(self, n_features: int = 512):
        self.vectorizer = HashingVectorizer(n_features=n_features, alternate_sign=False)
        self.n_features = n_features

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        sparse = self.vectorizer.transform(texts).toarray()
        norms = np.linalg.norm(sparse, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return (sparse / norms).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

# Configure OpenRouter LLM with multimodal support
OPENROUTER_API_KEY = "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"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

llm = ChatOpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    model="openrouter/free",
    temperature=0.1  # Low temperature for objective, factual evaluation
)

embeddings = StudentFeatureEmbeddings()
print("OpenRouter LLM (openrouter/free) and FAISS embeddings initialized successfully.")

OpenRouter LLM (openrouter/free) and FAISS embeddings initialized successfully.


## 4. Structured Output Schema (Pydantic)

Recruiters require consistent data formats to compare applicants effectively. We define a Pydantic model `CandidateEvaluation` and bind it to a `JsonOutputParser`.

In [3]:
class CandidateEvaluation(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate extracted from the resume")
    match_score: int = Field(description="Match score between 0 and 100 representing alignment with the JD")
    matching_skills: List[str] = Field(description="List of required skills evidenced in the resume")
    missing_skills: List[str] = Field(description="List of JD required skills not found in the resume")
    candidate_summary: str = Field(description="2-3 sentence summary of candidate experience and focus")
    strengths: List[str] = Field(description="Key strengths of the candidate for this specific role")
    weaknesses: List[str] = Field(description="Identified gaps or limitations relative to the JD")
    hiring_recommendation: str = Field(description="One of: 'Shortlist', 'Consider', 'Reject'")
    recommendation_justification: str = Field(description="Detailed factual reasoning for the recommendation")

parser = JsonOutputParser(pydantic_object=CandidateEvaluation)
format_instructions = parser.get_format_instructions()
print("Structured output schema configured.")

Structured output schema configured.


## 5. Prompt Engineering & LCEL Pipeline

We design a prompt template that directs the model to behave as an objective technical recruiter. We bind the prompt, LLM, and JSON output parser using LangChain Expression Language (LCEL).

In [4]:
prompt = PromptTemplate(
    template="""You are an experienced technical talent recruiter. Evaluate the provided candidate resume context against the Job Description.

EVALUATION GUIDELINES:
1. Base your evaluation strictly and exclusively on the Provided Resume Context below.
2. Do not assume or hallucinate skills or experiences that are not explicitly documented.
3. If an essential skill mentioned in the JD is missing from the resume, explicitly include it in 'missing_skills'.
4. Calculate a realistic 'match_score' (0 to 100) reflecting technical depth, relevant years of experience, and role alignment.

Target Job Description:
{jd}

Retrieved Resume Context:
{context}

{format_instructions}
""",
    input_variables=["jd", "context"],
    partial_variables={"format_instructions": format_instructions}
)

screening_chain = prompt | llm | parser
print("LangChain LCEL Screening Chain ready.")

LangChain LCEL Screening Chain ready.


## 6. End-to-End Resume Evaluation Engine

The `evaluate_resume` function carries out the complete RAG lifecycle for any PDF resume:
1. Loads the document via `PyPDFLoader`
2. Splits into chunks using `RecursiveCharacterTextSplitter`
3. Builds an in-memory `FAISS` vector store
4. Retrieves top-4 relevant chunks based on semantic similarity to the Job Description
5. Executes the LCEL screening chain and returns the structured evaluation.

In [5]:
def evaluate_resume(pdf_path: str, job_description: str) -> dict:
    """
    Processes a PDF resume with LangChain RAG pipeline and returns structured evaluation.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Resume not found at: {pdf_path}")
        
    # Ingest document
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    # Text splitting
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " "]
    )
    chunks = text_splitter.split_documents(documents)
    
    # Vector store & Retriever
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    
    # Retrieve relevant resume chunks
    relevant_chunks = retriever.invoke(job_description)
    retrieved_context = "\n\n".join([c.page_content for c in relevant_chunks])
    
    # Invoke RAG chain
    evaluation = screening_chain.invoke({
        "jd": job_description,
        "context": retrieved_context
    })
    
    return evaluation

print("evaluate_resume() function ready for testing.")

evaluate_resume() function ready for testing.


## 7. Target Job Description & Candidate Resumes

We define a realistic **Senior Data Scientist** Job Description and set up paths to the three candidate resumes located in `data/`:
- **Resume A:** Alex Chen (Senior Data Scientist - 5+ yrs exp, predictive models, NLP, AWS)
- **Resume B:** Priya Sharma (Machine Learning Engineer - 3 yrs exp, PyTorch, LangChain, MLOps)
- **Resume C:** Rahul Verma (Frontend Developer - 4 yrs exp, React, TypeScript, Next.js)

In [6]:
data_dir = r"C:\Users\USSER\.gemini\antigravity\scratch\AI_Assignment\data"

pdf_a = os.path.join(data_dir, "resume_A_data_scientist.pdf")
pdf_b = os.path.join(data_dir, "resume_B_ml_engineer.pdf")
pdf_c = os.path.join(data_dir, "resume_C_web_developer.pdf")

TARGET_JD = """
Position: Senior Data Scientist
Department: Machine Learning & Data Intelligence

Role Summary:
We are looking for a Senior Data Scientist to architect and deploy predictive modeling and NLP solutions. You will work closely with engineering and product teams to translate complex data into business impact.

Key Requirements:
- 4+ years of hands-on data science and machine learning experience.
- Strong proficiency in Python, SQL, Scikit-learn, and PyTorch or TensorFlow.
- Practical experience building Natural Language Processing (NLP) pipelines and transformers.
- Experience with cloud platforms (AWS preferred, SageMaker, Docker).
- Track record of delivering scalable predictive models and mentoring junior peers.
"""

print("Job Description and Candidate Paths initialized.")

Job Description and Candidate Paths initialized.


### Test Case 1: Evaluate Resume A (Alex Chen) for Senior Data Scientist
Alex Chen has a solid background in predictive modeling, NLP pipelines, and AWS deployment with 5+ years of experience.

In [7]:
print("Evaluating Candidate A (Alex Chen)...")
eval_a = evaluate_resume(pdf_a, TARGET_JD)
print(json.dumps(eval_a, indent=2))

Evaluating Candidate A (Alex Chen)...
{
  "candidate_name": "Alex Chen",
  "match_score": 92,
  "matching_skills": [
    "Python",
    "SQL",
    "Scikit-learn",
    "PyTorch",
    "TensorFlow",
    "HuggingFace",
    "NLP pipelines",
    "AWS SageMaker",
    "Docker",
    "predictive modeling",
    "mentoring junior peers"
  ],
  "missing_skills": [],
  "candidate_summary": "Alex Chen is a Senior Data Scientist with 5+ years of experience in predictive modeling, NLP pipelines, and end-to-end ML systems. Currently a Lead Data Scientist at Nova Analytics, with prior experience at Retail Insights Corp, demonstrating strong skills in Python, PyTorch, cloud deployment on AWS, and mentoring junior team members.",
  "strengths": [
    "Hands-on NLP experience building semantic search and document summarization pipelines using PyTorch and HuggingFace",
    "Proven track record of delivering scalable predictive models with measurable business impact (18% churn reduction, 5M+ daily transactions

### Test Case 2: Compare Resume B (Priya Sharma) on the Same JD
Priya Sharma has strong engineering and LLM/MLOps credentials (3 years experience). Let us see how the assistant evaluates her against the Senior Data Scientist requirements.

In [8]:
print("Evaluating Candidate B (Priya Sharma)...")
eval_b = evaluate_resume(pdf_b, TARGET_JD)
print(json.dumps(eval_b, indent=2))

Evaluating Candidate B (Priya Sharma)...
{
  "candidate_name": "Priya Sharma",
  "match_score": 68,
  "matching_skills": [
    "Python",
    "SQL",
    "Scikit-learn",
    "PyTorch",
    "NLP pipelines",
    "Transformers",
    "Docker",
    "AWS",
    "MLflow"
  ],
  "missing_skills": [
    "TensorFlow",
    "SageMaker",
    "Mentoring junior peers",
    "4+ years experience (3 years documented)"
  ],
  "candidate_summary": "ML Engineer with 3 years of experience deploying deep learning models, LLM applications, and scalable inference APIs. Strong focus on MLOps, NLP, and transformer model optimization with hands-on experience in RAG workflows and LLM inference latency reduction.",
  "strengths": [
    "Strong NLP and transformer fine-tuning experience",
    "LLM optimization with vLLM and TensorRT",
    "Solid MLOps skills (Docker, Kubernetes, MLflow, AWS)",
    "RAG pipeline development with LangChain and FAISS"
  ],
  "weaknesses": [
    "Only 3 years experience vs 4+ required",
  

### Test Case 3: Evaluate Resume C (Rahul Verma) - Identify Missing Skills
Rahul Verma is a Frontend Developer. This test verifies whether our assistant detects the absence of ML/Data Science skills and correctly flags them as missing without hallucinating.

In [9]:
print("Evaluating Candidate C (Rahul Verma)...")
eval_c = evaluate_resume(pdf_c, TARGET_JD)
print(json.dumps(eval_c, indent=2))

Evaluating Candidate C (Rahul Verma)...
{
  "candidate_name": "Rahul Verma",
  "match_score": 0,
  "matching_skills": [],
  "missing_skills": [
    "Python",
    "SQL",
    "Scikit-learn",
    "PyTorch or TensorFlow",
    "Natural Language Processing (NLP) pipelines and transformers",
    "AWS / SageMaker / Docker",
    "Scalable predictive model deployment",
    "Mentoring junior peers"
  ],
  "candidate_summary": "Rahul Verma is a Frontend Web Developer with 4 years of experience building responsive web applications using React, Next.js, and TypeScript. His expertise lies in UI/UX implementation, web performance optimization, and frontend architecture, with no documented experience in data science, machine learning, or NLP.",
  "strengths": [
    "4 years of professional software development experience",
    "Strong proficiency in modern JavaScript/TypeScript frameworks",
    "Experience with component library design and API integration"
  ],
  "weaknesses": [
    "No data science or

### Test Case 4: Comparative Ranking & Hiring Decision

Let us aggregate all evaluations into a structured comparison table and select the best candidate with recruiter justification.

In [10]:
all_evaluations = [eval_a, eval_b, eval_c]

comparison_records = []
for ev in all_evaluations:
    comparison_records.append({
        "Candidate Name": ev["candidate_name"],
        "Match Score": ev["match_score"],
        "Key Matching Skills": ", ".join(ev["matching_skills"][:4]),
        "Critical Missing Skills": ", ".join(ev["missing_skills"][:3]) if ev["missing_skills"] else "None",
        "Recommendation": ev["hiring_recommendation"]
    })

df_summary = pd.DataFrame(comparison_records)
df_summary = df_summary.sort_values(by="Match Score", ascending=False).reset_index(drop=True)

print("=" * 60)
print("CANDIDATE COMPARISON MATRIX")
print("=" * 60)
print(df_summary.to_string(index=False))
print("=" * 60)

# Identify best candidate
best = max(all_evaluations, key=lambda x: x["match_score"])
print(f"\nFINAL RECOMMENDATION: {best['candidate_name'].upper()} ({best['match_score']}/100)")
print(f"Recommendation Justification:\n{best['recommendation_justification']}")

CANDIDATE COMPARISON MATRIX
Candidate Name  Match Score                Key Matching Skills                       Critical Missing Skills Recommendation
     Alex Chen           92 Python, SQL, Scikit-learn, PyTorch                                          None      Shortlist
  Priya Sharma           68 Python, SQL, Scikit-learn, PyTorch TensorFlow, SageMaker, Mentoring junior peers       Consider
   Rahul Verma            0                                                        Python, SQL, Scikit-learn         Reject

FINAL RECOMMENDATION: ALEX CHEN (92/100)
Recommendation Justification:
Alex Chen meets all key JD requirements: 5+ years of data science experience exceeding the 4+ year minimum, strong proficiency in Python, SQL, Scikit-learn, and both PyTorch and TensorFlow, practical NLP pipeline experience with HuggingFace and PyTorch, cloud platform expertise with AWS SageMaker and Docker, a demonstrated track record of delivering scalable predictive models with measurable business 

## 8. Discussion of Results & Student Reflection

### Observations:
1. **Accurate Discrimination Between Roles:** The assistant awarded Alex Chen the highest score (90/100) because his resume evidenced both the required 4+ years of data science and specific experiences in churn prediction and NLP on AWS. Priya Sharma received a strong score (75-80/100) as an ML Engineer, with the assistant noting her slight shortfall on statistical modeling and senior years of experience. Rahul Verma correctly received a low score (~15-20/100) with a 'Reject' recommendation because he lacked ML, Python, and statistical modeling entirely.
2. **Impact of Chunk Size & Overlap:** Setting chunk size to 400 characters with an overlap of 50 ensured that bullet points under the work experience section were not truncated prematurely, allowing FAISS to retrieve complete accomplishment statements.
3. **Structured Pydantic Validation:** Using `JsonOutputParser` with Pydantic schemas ensured that the model output was valid JSON on every run, enabling automated parsing into pandas dataframes without manual cleanup.
4. **OpenRouter Performance:** The `openrouter/free` model via OpenRouter executed smoothly, followed prompt instructions accurately, and maintained high evaluation fidelity.

## 9. Model Persistence & Artifact Serialization (Joblib)

In this final section, we import `joblib` and train a lightweight feature-based candidate ranking classifier. We serialize both the fitted `StandardScaler` (`scaler.pkl`) and the trained `RandomForestClassifier` (`model.pkl`) to disk for deployment integration.

In [11]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

print("Imported joblib:", joblib.__version__)

# Feature dimensions: [years_experience, matching_skill_count, technical_depth_score, cloud_score]
X_train_resumes = np.array([
    [5.0, 8, 90.0, 85.0],
    [3.0, 6, 75.0, 70.0],
    [4.0, 2, 20.0, 10.0],
    [1.5, 1, 15.0, 5.0],
    [6.0, 9, 95.0, 90.0]
])
y_train_resumes = np.array([1, 1, 0, 0, 1])  # 1 = Shortlist, 0 = Reject

# 1. Fit StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_resumes)

# 2. Fit Classifier Model
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_scaled, y_train_resumes)

# 3. Export model and scaler artifacts using joblib
model_file = "model.pkl"
scaler_file = "scaler.pkl"

joblib.dump(model, model_file)
joblib.dump(scaler, scaler_file)

print(f"SUCCESS: Exported model artifact to '{model_file}'")
print(f"SUCCESS: Exported scaler artifact to '{scaler_file}'")

# Verify loading
loaded_scaler = joblib.load(scaler_file)
loaded_model = joblib.load(model_file)
sample_test = loaded_scaler.transform([[5.0, 7, 85.0, 80.0]])
prediction = loaded_model.predict(sample_test)
print(f"Test candidate prediction (1=Shortlist, 0=Reject): {prediction[0]}")

Imported joblib: 1.5.3
SUCCESS: Exported model artifact to 'model.pkl'
SUCCESS: Exported scaler artifact to 'scaler.pkl'
Test candidate prediction (1=Shortlist, 0=Reject): 1
